In [ ]:
!pip install -q scikit-learn xgboost imbalanced-learn joblib
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib
import os

PROJECT_DIR = '/content/drive/MyDrive/Ip-Masked'
MODEL_DIR = f'{PROJECT_DIR}/models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(f'{PROJECT_DIR}/data/processed/features_dataset.csv')

FEATURES = [
    'ip_version', 'is_private', 'is_reserved', 'is_loopback',
    'is_multicast', 'octet_1', 'octet_2', 'octet_3', 'octet_4',
    'latitude', 'longitude', 'accuracy_radius', 'asn',
    'has_ptr_record', 'ptr_contains_host', 'in_tor_list',
    'in_proxy_list', 'in_vpn_list', 'request_count',
    'unique_user_agents'
]

X = df[FEATURES]
y = df['label']

print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]

print('Random Forest Results')
print(classification_report(y_test, rf_preds))
print('ROC-AUC:', roc_auc_score(y_test, rf_probs))

In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_test)
xgb_probs = xgb.predict_proba(X_test)[:, 1]

print('XGBoost Results')
print(classification_report(y_test, xgb_preds))
print('ROC-AUC:', roc_auc_score(y_test, xgb_probs))

In [ ]:
joblib.dump(rf, f'{MODEL_DIR}/random_forest_model.pkl')
joblib.dump(xgb, f'{MODEL_DIR}/xgboost_model.pkl')

print('Models saved successfully')